# Search Quality Evaluation Tutorial

This notebook demonstrates how to evaluate **search quality** - measuring query effectiveness, result coverage, and credit efficiency.

**Key metrics:**
- Query quality score
- Result coverage
- Deduplication efficiency
- Credit efficiency

## Setup

In [1]:
%pip install -q tavily-agent-toolkit

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("TAVILY_API_KEY:\n")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY:\n")

In [ ]:
from tavily import TavilyClient
from tavily_agent_toolkit import ModelConfig, ModelObject
from tavily_agent_toolkit.evals import compute_search_quality_metrics

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

judge_config = ModelConfig(
    model=ModelObject(
        model="gpt-5-mini",
        api_key=os.environ["OPENAI_API_KEY"],
    ),
)

## Example 1: Evaluating Multi-Query Search

In [5]:
# Research task requiring multiple search queries
research_task = "Compare the energy efficiency and cost of solar panels vs wind turbines for residential use"

# Generated search queries (could come from query expansion)
queries = [
    "solar panel energy efficiency residential 2024",
    "wind turbine home installation cost",
    "solar vs wind power residential comparison",
]

# Execute searches and collect results
all_results = []
total_credits = 0

for query in queries:
    response = tavily_client.search(
        query=query,
        max_results=5,
        include_usage=True,
    )
    all_results.extend(response["results"])
    total_credits += response.get("usage", {}).get("credits", 0)

print(f"Executed {len(queries)} queries")
print(f"Retrieved {len(all_results)} results")
print(f"Used {total_credits} credits")

Executed 3 queries
Retrieved 15 results
Used 3 credits


In [6]:
# Evaluate search quality
quality = await compute_search_quality_metrics(
    research_task=research_task,
    queries=queries,
    results=all_results,
    judge_model_config=judge_config,
    credits_used=total_credits,
)

print("Search Quality Metrics:")
print("="*50)
print(f"Query Quality:       {quality.query_quality_score:.2f}")
print(f"Result Coverage:     {quality.result_coverage:.1%}")
print(f"Dedup Efficiency:    {quality.deduplication_efficiency:.1%}")
print(f"Credit Efficiency:   {quality.credit_efficiency:.4f}")
print(f"\nUnique Results: {quality.unique_results_count}/{quality.results_count}")

Search Quality Metrics:
Query Quality:       0.68
Result Coverage:     60.0%
Dedup Efficiency:    100.0%
Credit Efficiency:   0.2133

Unique Results: 15/15


In [7]:
# View query-level feedback
print("\nQuery Analysis:")
for qd in quality.query_details:
    print(f"\n  Query: {qd['query']}")
    print(f"    Specificity: {qd['specificity']:.2f}")
    print(f"    Effectiveness: {qd['search_effectiveness']:.2f}")
    if qd.get('suggestions'):
        print(f"    Suggestions: {', '.join(qd['suggestions'][:2])}")


Query Analysis:

  Query: solar panel energy efficiency residential 2024
    Specificity: 0.80
    Effectiveness: 0.80
    Suggestions: Specify which efficiency metric you mean (module efficiency, system efficiency, performance ratio, or kWh produced per kW installed)., Add location/region or climate (e.g., 'US', 'UK', 'California') because residential performance depends on insolation and policy.

  Query: wind turbine home installation cost
    Specificity: 0.70
    Effectiveness: 0.75
    Suggestions: Clarify scale: use 'small wind turbine', 'residential-scale', or specify capacity (e.g., '1–10 kW')., Add region or country because installation, permitting, and grid connection costs vary widely.

  Query: solar vs wind power residential comparison
    Specificity: 0.60
    Effectiveness: 0.65
    Suggestions: Make the comparison metrics explicit: add 'energy efficiency', 'installation cost', 'LCOE', 'capacity factor', 'kWh/year', or 'payback period'., Include location/climate and sy

## Example 2: Optimizing Query Strategy

In [8]:
# Compare different query strategies
task = "Latest developments in mRNA vaccine technology"

# Strategy 1: Single broad query
single_query = ["mRNA vaccine developments 2024"]
single_results = tavily_client.search(query=single_query[0], max_results=10, include_usage=True)

# Strategy 2: Multiple specific queries
multi_queries = [
    "mRNA vaccine clinical trials 2024",
    "mRNA technology new applications beyond COVID",
    "mRNA manufacturing advances",
]
multi_results = []
for q in multi_queries:
    r = tavily_client.search(query=q, max_results=5)
    multi_results.extend(r["results"])

In [9]:
# Evaluate both strategies
single_quality = await compute_search_quality_metrics(
    research_task=task,
    queries=single_query,
    results=single_results["results"],
    judge_model_config=judge_config,
)

multi_quality = await compute_search_quality_metrics(
    research_task=task,
    queries=multi_queries,
    results=multi_results,
    judge_model_config=judge_config,
)

print("Strategy Comparison")
print("="*60)
print(f"{'Metric':<25} {'Single Query':<18} {'Multi-Query':<18}")
print(f"{'-'*25} {'-'*18} {'-'*18}")
print(f"{'Query Quality':<25} {single_quality.query_quality_score:<18.2f} {multi_quality.query_quality_score:<18.2f}")
print(f"{'Coverage':<25} {single_quality.result_coverage:<18.1%} {multi_quality.result_coverage:<18.1%}")
print(f"{'Results':<25} {single_quality.results_count:<18} {multi_quality.results_count:<18}")
print(f"{'Unique Results':<25} {single_quality.unique_results_count:<18} {multi_quality.unique_results_count:<18}")

Strategy Comparison
Metric                    Single Query       Multi-Query       
------------------------- ------------------ ------------------
Query Quality             0.45               0.73              
Coverage                  65.0%              70.0%             
Results                   9                  15                
Unique Results            9                  15                


## Interpreting Results

### Query Quality Score
How effective are your search queries?
- **0.8+**: Excellent queries, specific and search-optimized
- **0.6-0.8**: Good queries with room for improvement
- **<0.6**: Queries may be too vague or poorly constructed

### Result Coverage
How well do results cover the research topic?

### Deduplication Efficiency
Unique results / total results. Low values indicate redundant searching.

### Credit Efficiency
Quality score / credits used. Higher = better value per credit.

In [ ]:
# Export results
print(quality.to_dict())